In [ ]:
!pip install llama-index-core llama-index-llms-openai llama-index-embeddings-openai

llama-index-core → LlamaIndex 핵심 기능

llama-index-llms-openai → OpenAI LLM 연동

llama-index-embeddings-openai → OpenAI 임베딩 연동

In [ ]:
#기본 유닛 테스트
from llama_index.core.llms import MockLLM
from llama_index.core import MockEmbedding, Settings, VectorStoreIndex
from llama_index.core.schema import Document

Settings.llm = MockLLM(max_tokens=256)
Settings.embed_model = MockEmbedding(embed_dim=1536)

def test_index_creation():
    docs = [Document(text="서울의 날씨는 맑습니다.")]
    index = VectorStoreIndex.from_documents(docs)
    assert index is not None
    print("✅ test_index_creation 통과")

def test_query_engine_returns_response():
    docs = [Document(text="파이썬은 프로그래밍 언어입니다.")]
    index = VectorStoreIndex.from_documents(docs)
    engine = index.as_query_engine()
    response = engine.query("파이썬이 뭐야?")
    assert response is not None
    assert hasattr(response, "response")
    assert hasattr(response, "source_nodes")
    print("✅ test_query_engine_returns_response 통과")

test_index_creation()
test_query_engine_returns_response()

"RAG 파이프라인 구조 자체가 정상적으로 동작하는가?" 를 테스트한것. API 호출 없이.

test_index_creation       → 문서를 인덱스로 만들 수 있는가?
test_query_engine_returns_response → 쿼리 엔진이 응답 객체를 돌려주는가?
실제 답변 내용이 맞는지는 안 봐요. "구조가 깨지지 않았는가" 만 확인하는 거예요.


In [ ]:
# 토큰 사용량 테스트
import tiktoken
from llama_index.core.callbacks import CallbackManager, TokenCountingHandler
from llama_index.core.llms import MockLLM
from llama_index.core import MockEmbedding, Settings, VectorStoreIndex
from llama_index.core.schema import Document

def test_token_counting():
    token_counter = TokenCountingHandler(
        tokenizer=tiktoken.encoding_for_model("gpt-3.5-turbo").encode
    )
    Settings.llm = MockLLM(max_tokens=256)
    Settings.embed_model = MockEmbedding(embed_dim=1536)
    Settings.callback_manager = CallbackManager([token_counter])

    docs = [Document(text="LlamaIndex는 RAG 전문 프레임워크입니다.")]
    index = VectorStoreIndex.from_documents(docs)

    assert token_counter.total_embedding_token_count > 0
    print(f"임베딩 토큰: {token_counter.total_embedding_token_count}")
    print("✅ test_token_counting 통과")

test_token_counting()

셀 3 - 토큰 카운팅 테스트

"문서를 인덱싱할 때 토큰이 실제로 소비됐는가?" 를 테스트한 거예요. 이것도 API 호출 없이요.

pythontoken_counter = TokenCountingHandler(...)  # 토큰 수 세는 도구

"LlamaIndex는 RAG 전문 프레임워크입니다." → 이 텍스트가 벡터로 변환될 때
몇 개의 토큰이 사용됐는지 카운팅

In [ ]:
#FaithfulnessEvaluator - 환각 여부 테스트
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

from llama_index.core import VectorStoreIndex, Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.evaluation import FaithfulnessEvaluator
from llama_index.core.schema import Document

Settings.llm = OpenAI(model="gpt-3.5-turbo", temperature=0.0)  # 비용 절감용
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

evaluator = FaithfulnessEvaluator(llm=Settings.llm)

documents = [
    Document(text="서울의 인구는 약 950만 명으로, 대한민국에서 가장 큰 도시입니다.")
]
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine()

response = query_engine.query("서울의 인구는 얼마야?")
eval_result = evaluator.evaluate_response(response=response)

print(f"통과 여부: {eval_result.passing}")
print(f"피드백: {eval_result.feedback}")
print("✅ FaithfulnessEvaluator 테스트 완료")

전체 RAG 파이프라인을 실제로 돌린 코드.

① 문서 등록
"서울의 인구는 약 950만 명..."

② 인덱싱 (OpenAI 임베딩으로 벡터화)

③ 질문: "서울의 인구는 얼마야?"

④ 벡터 검색으로 관련 문서 찾기

⑤ LLM(gpt-3.5-turbo)이 문서 기반으로 답변 생성

⑥ FaithfulnessEvaluator: "이 답변이 문서에 근거한 건가?" 판단

passing: True / False

문서에 "950만 명"이라고 있는데, LLM이 "1500만 명"이라고 답하면 → False (환각)
문서 내용 그대로 답하면 → True (충실)

# 아래부터는 시나리오 반영한 코드
사용자 시나리오
페르소나: 대학생 김민준 (22세)

시험 공부하다가 두통이 생김
집에 상비약 여러 개 있지만 뭘 먹어야 할지 모름
약학 지식 없음, 약 설명서 읽기 귀찮음
"그냥 챗봇한테 물어보자" 마음으로 질문

테스트 질문 3가지:

"머리가 너무 아픈데 타이레놀 먹어도 돼?"
"타이레놀 한 번에 몇 알 먹어야 해?"
"밥 안 먹었는데 타이레놀 먹어도 괜찮아?"

In [ ]:
# 셀 1 - 설치
!pip install llama-index-core llama-index-llms-openai llama-index-embeddings-openai

In [ ]:
# 셀 2 - API 키 설정
from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
print("✅ API 키 설정 완료")

In [ ]:
# 셀 3 - 타이레놀 문서 등록 + 인덱싱
from llama_index.core import VectorStoreIndex, Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.schema import Document

Settings.llm = OpenAI(model="gpt-3.5-turbo", temperature=0.0)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

# 타이레놀 문서 (설명서 기반)
tylenol_doc = Document(text="""
[타이레놀 (아세트아미노펜) 설명서]

■ 효능·효과
두통, 치통, 생리통, 근육통, 발열 완화

■ 용법·용량
- 성인 및 15세 이상: 1회 1~2정 (500mg~1000mg)
- 1일 3~4회 복용 가능
- 복용 간격: 최소 4~6시간 이상
- 1일 최대 용량: 4000mg (8정) 초과 금지

■ 공복 복용
공복에 복용해도 위장 자극이 적어 괜찮습니다.
단, 위장이 예민한 경우 소량의 음식과 함께 복용 권장.

■ 주의사항
- 간 질환자, 알코올 과다 복용자는 복용 전 의사 상담 필요
- 음주 중 또는 음주 후 복용 주의 (간 손상 위험)
- 다른 아세트아미노펜 성분 의약품과 중복 복용 금지
- 임산부·수유부는 복용 전 의사 상담

■ 부작용
- 드물게 피부 발진, 두드러기 발생 가능
- 과량 복용 시 간 손상 위험
""", metadata={"source": "타이레놀 설명서", "drug": "타이레놀"})

# 인덱싱
print("📄 문서 인덱싱 중...")
index = VectorStoreIndex.from_documents([tylenol_doc])
print("✅ 인덱싱 완료\n")
print(f"등록된 문서 내용 미리보기:\n{tylenol_doc.text[:100]}...")

In [ ]:
# 셀 4 - 쿼리 엔진 설정 (추론 과정 출력 포함)
query_engine = index.as_query_engine(
    similarity_top_k=3,   # 상위 3개 관련 문서 청크 검색
    verbose=True          # 내부 추론 과정 출력
)
print("✅ 쿼리 엔진 준비 완료")

In [ ]:
# 셀 5 - 시나리오 테스트 실행
questions = [
    "머리가 너무 아픈데 타이레놀 먹어도 돼? 타이레놀 한 번에 몇 알 먹어야 해?",
    "타이레놀 말고 다른거 두통약 찾아줘",
    "밥 안 먹었는데 타이레놀 먹어도 괜찮아?"
]

for i, question in enumerate(questions, 1):
    print("=" * 60)
    print(f"[질문 {i}] {question}")
    print("=" * 60)

    response = query_engine.query(question)

    # 검색된 소스 문서 출력
    # 유사도 점수 : 질문과 문서가 얼마나 관련 있는지 나타내는 숫자.
    # 1에 가까울 수록 관련도 높음
    print("\n📚 검색된 소스 문서:")
    for j, node in enumerate(response.source_nodes, 1):
        print(f"  [{j}] 유사도 점수: {node.score:.4f}")
        print(f"       내용: {node.text[:80]}...")

    # LLM 답변 출력
    print(f"\n🤖 LLM 답변:\n{response.response}")
    print()

In [ ]:
# 셀 6 - FaithfulnessEvaluator로 환각 검사
from llama_index.core.evaluation import FaithfulnessEvaluator

evaluator = FaithfulnessEvaluator(llm=Settings.llm)

print("🔍 환각(Hallucination) 검사 시작\n")

for i, question in enumerate(questions, 1):
    print(f"[질문 {i}] {question}")

    response = query_engine.query(question)
    eval_result = evaluator.evaluate_response(response=response)

    status = "✅ 통과 (문서 기반 답변)" if eval_result.passing else "❌ 실패 (환각 의심)"
    print(f"  결과: {status}")
    print(f"  판단 근거: {eval_result.feedback}")
    print()

# 프롬프트 입력 (셀 4에서부터 진행)

In [ ]:
# 셀 4 - 프롬프트 커스터마이징 + 쿼리 엔진 설정
from llama_index.core.prompts import PromptTemplate

# 커스텀 프롬프트 작성
custom_prompt = PromptTemplate(
    """당신은 친근한 고양이 약사 챗봇입니다.
아래 문서 내용을 참고해서 사용자 질문에 답변해주세요.

규칙:
- 반드시 문서에 있는 내용만 바탕으로 답변할 것
- 모든 문장 끝에 '냥'과 고양이 이모티콘(🐱)을 붙일 것
- 예: "타이레놀은 두통에 효과적이냥 🐱"
- 약학 지식이 없는 대학생도 이해할 수 있게 쉽게 설명할 것
- 주의사항이 있으면 꼭 포함할 것

---------------------
참고 문서:
{context_str}
---------------------

질문: {query_str}

답변:"""
)

query_engine = index.as_query_engine(
    similarity_top_k=3,
    text_qa_template=custom_prompt,  # 커스텀 프롬프트 주입
    verbose=True
)

print("✅ 커스텀 프롬프트 적용된 쿼리 엔진 준비 완료")
print("\n적용된 프롬프트 미리보기:")
print(custom_prompt.template[:150] + "...")

In [ ]:
# 셀 5 - 시나리오 테스트 실행
questions = [
    "머리가 너무 아픈데 타이레놀 먹어도 돼?",
    "타이레놀 한 번에 몇 알 먹어야 해?",
    "밥 안 먹었는데 타이레놀 먹어도 괜찮아?"
]

for i, question in enumerate(questions, 1):
    print("=" * 60)
    print(f"[질문 {i}] {question}")
    print("=" * 60)

    response = query_engine.query(question)

    # 검색된 소스 문서 출력
    print("\n📚 검색된 소스 문서:")
    for j, node in enumerate(response.source_nodes, 1):
        print(f"  [{j}] 유사도 점수: {node.score:.4f}")
        print(f"       내용: {node.text[:80]}...")

    # LLM 답변 출력
    print(f"\n🤖 LLM 답변:\n{response.response}")
    print()

In [ ]:
# 셀 6 - FaithfulnessEvaluator로 환각 검사
from llama_index.core.evaluation import FaithfulnessEvaluator

evaluator = FaithfulnessEvaluator(llm=Settings.llm)

print("🔍 환각(Hallucination) 검사 시작\n")

for i, question in enumerate(questions, 1):
    print(f"[질문 {i}] {question}")

    response = query_engine.query(question)
    eval_result = evaluator.evaluate_response(response=response)

    status = "✅ 통과 (문서 기반 답변)" if eval_result.passing else "❌ 실패 (환각 의심)"
    print(f"  결과: {status}")
    print(f"  판단 근거: {eval_result.feedback}")
    print()